#### **Importation des bibliothèques**

In [ ]:
# Utilitaires de base
import builtins

# Suivi des expriences (MLflow & DagsHub)
import dagshub
import mlflow

# Scikit-Learn : Sparation des donnes et mtriques d'valuation

# TensorFlow / Keras : Cration et entranement du modle Deep Learning

#### **DagsHub & MLflow Init**

In [ ]:

# Initialisation de la connexion DagsHub avec les identifiants de votre dépôt et activation du mode MLflow
# PATCH WINDOWS : Force l'utilisation de l'encodage UTF-8 lors de l'écriture des fichiers.
# Ceci corrige l'erreur "charmap codec can't encode characters" causée par le nouveau format d'affichage (summary) de Keras 3
_original_open = builtins.open
def _utf8_open(*args, **kwargs):
    mode = kwargs.get('mode', args[1] if len(args) > 1 else 'r')
    if 'b' not in mode and 'encoding' not in kwargs:
        kwargs['encoding'] = 'utf-8'
    return _original_open(*args, **kwargs)
builtins.open = _utf8_open

dagshub.init(repo_owner='Oscar-AS', repo_name='disaster-tweets-project', mlflow=True)

# Définition du nom du dossier (expérience) dans MLflow où toutes nos métriques seront classées
mlflow.set_experiment("Disaster_Tweets_Niveau_3_et_4")

# Affichage d'un message console pour confirmer que le tracking est bien connecté
print("MLflow activé avec succès sur DagsHub !")


#### **Sélection du Meilleur Modèle (Mise en Production)**
Nous interrogeons l'historique MLflow pour trouver le meilleur modèle selon le **F2-Score** et le **Rappel (Classe 1)**, puis nous l'enregistrons dans le Model Registry.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

# 1. Obtenir l'ID de notre expérience
experiment = mlflow.get_experiment_by_name("Disaster_Tweets_Niveau_3_et_4")

if experiment is not None:
    # 2. Récupérer tous les runs de cette expérience
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
    
    # On s'assure d'avoir des résultats
    if not runs.empty and 'metrics.eval_f2_score' in runs.columns:
        # 3. Trier les modèles. 
        # Critère 1 : Le plus haut F2-Score (qui privilégie le rappel)
        # Critère 2 : Le plus haut Rappel sur la classe 1 (les vrais désastres) en cas d'égalité
        sorted_runs = runs.sort_values(
            by=['metrics.eval_f2_score', 'metrics.eval_recall_class_1'], 
            ascending=[False, False]
        )
        
        # 4. Prendre le meilleur run
        best_run = sorted_runs.iloc[0]
        best_run_id = best_run['run_id']
        best_run_name = best_run['tags.mlflow.runName']
        best_f2 = best_run['metrics.eval_f2_score']
        best_recall = best_run['metrics.eval_recall_class_1']
        
        print("=== MEILLEUR MODÈLE TROUVÉ ===")
        print(f"Nom du Run : {best_run_name}")
        print(f"F2-Score : {best_f2:.4f}")
        print(f"Recall (Désastres) : {best_recall:.4f}")
        print(f"Run ID : {best_run_id}")
        print("==============================")
        
        # 5. Enregistrer ce modèle dans le Model Registry MLflow
        model_uri = f"runs:/{best_run_id}/model"
        model_name = "Disaster_Tweet_Predictor_Prod"
        
        try:
            print(f"\nEnregistrement du modèle '{model_name}' dans le MLflow Registry...")
            # Enregistrement
            registered_model = mlflow.register_model(model_uri=model_uri, name=model_name)
            
            # 6. Passer le modèle en phase de "Production"
            client = MlflowClient()
            client.transition_model_version_stage(
                name=model_name,
                version=registered_model.version,
                stage="Production",
                archive_existing_versions=True
            )
            print(f"Succès ! Le modèle version {registered_model.version} est maintenant en PRODUCTION.")
            print(f"Vous pouvez le charger plus tard avec : mlflow.pyfunc.load_model(f'models:/{model_name}/Production')")
            
        except Exception as e:
            print("Erreur lors de l'enregistrement dans le Registry. Si vous êtes en local sans serveur de tracking avancé, c'est normal.")
            print("Détail de l'erreur :", e)
    else:
        print("Aucune métrique 'eval_f2_score' trouvée. Assurez-vous d'avoir entraîné les modèles d'abord.")
else:
    print("L'expérience MLflow n'a pas été trouvée.")
